# LES turbulent flow past a sphere

This notebook runs a **Smagorinsky LES** finite-volume simulation of turbulent flow past the sphere in `fluid_mesh_3d.msh`.

## Boundary conditions (chosen not to disturb the wake)
- **Inlet**: fixed freestream $U_\infty$ + mild turbulence intensity
- **Outlet**: convective (Orlanski) outflow + soft pressure (non-reflecting)
- **Outer walls**: free-slip (reduces blockage vs no-slip channel walls)
- **Sphere (object)**: no-slip

Outputs: mid-plane velocity/pressure plots and a wake centerline profile.

In [ ]:
import os
print("Working directory:", os.getcwd())
print("Files:", [f for f in os.listdir() if f.endswith(('.npz', '.py', '.msh', '.png'))])

In [ ]:
# If needed:
# %pip install numpy scipy matplotlib meshio

## 1) Ensure processed mesh includes boundary tags

`processed_mesh.npz` must contain `boundary_tag` (and patch face lists).  
If missing, rebuild tags from `fluid_mesh_3d.msh`:

In [ ]:
import numpy as np
import meshio

mesh_npz = np.load("processed_mesh.npz")
need_rebuild = "boundary_tag" not in mesh_npz.files

if need_rebuild:
    print("Rebuilding boundary tags into processed_mesh.npz ...")
    owner = mesh_npz["owner"]
    neighbour = mesh_npz["neighbour"]
    unique_faces = mesh_npz["unique_faces"]
    face_lookup = {tuple(f): i for i, f in enumerate(unique_faces)}

    m = meshio.read("fluid_mesh_3d.msh")
    physical_names = {1: "inlet", 2: "outlet", 3: "walls", 4: "object"}
    boundary_faces = {k: [] for k in physical_names.values()}
    boundary_tag = np.full(len(unique_faces), 5, dtype=np.int32)

    for block, tags in zip(m.cells[:-1], m.cell_data["gmsh:physical"][:-1]):
        for tri, tag in zip(block.data, tags):
            fi = face_lookup[tuple(sorted(tri))]
            boundary_faces[physical_names[int(tag)]].append(fi)
            boundary_tag[fi] = int(tag)

    np.savez(
        "processed_mesh.npz",
        points=mesh_npz["points"],
        tetra=mesh_npz["tetra"],
        cell_centroids=mesh_npz["cell_centroids"],
        cell_volumes=mesh_npz["cell_volumes"],
        unique_faces=unique_faces,
        owner=owner,
        neighbour=neighbour,
        face_centroids=mesh_npz["face_centroids"],
        face_area_vectors=mesh_npz["face_area_vectors"],
        face_areas=mesh_npz["face_areas"],
        face_normals=mesh_npz["face_normals"],
        boundary_tag=boundary_tag,
        inlet_faces=np.array(boundary_faces["inlet"], dtype=np.int32),
        outlet_faces=np.array(boundary_faces["outlet"], dtype=np.int32),
        wall_faces=np.array(boundary_faces["walls"], dtype=np.int32),
        object_faces=np.array(boundary_faces["object"], dtype=np.int32),
    )
    for k, v in boundary_faces.items():
        print(f"  {k}: {len(v)} faces")
else:
    tag = mesh_npz["boundary_tag"]
    print("boundary_tag already present")
    print("  inlet ", np.sum(tag == 1))
    print("  outlet", np.sum(tag == 2))
    print("  walls ", np.sum(tag == 3))
    print("  object", np.sum(tag == 4))

## 2) Run the LES solver

Use `quick=True` for a short demo, or increase `t_end` / `max_steps` for a longer wake development.

In [ ]:
from les_sphere_flow import SphereLESSolver, SimConfig

quick = True  # set False for a longer run

if quick:
    cfg = SimConfig(dt=1.0e-4, t_end=0.05, print_every=50, inlet_ti=0.015)
    max_steps = 200
else:
    cfg = SimConfig(dt=1.0e-4, t_end=0.5, print_every=100, inlet_ti=0.015)
    max_steps = None

solver = SphereLESSolver(mesh_path="processed_mesh.npz", cfg=cfg)
solver.run(max_steps=max_steps)
solver.save_fields("les_sphere_fields.npz")

## 3) Visualization — turbulent flow past the sphere

In [ ]:
from IPython.display import Image, display

mid = solver.plot_midplane("flow_past_sphere_midplane.png")
wake = solver.plot_wake_profile("flow_past_sphere_wake.png")

display(Image(filename=mid))
display(Image(filename=wake))

## Notes
- Geometry is a **sphere** (not a cylinder) in a 10×5×5 box; centre `(3, 2.5, 2.5)`, radius `0.5`.
- Implied Reynolds number: $Re = U_\infty D / \nu \approx 1000$ with defaults.
- Soft inlet/outlet BCs are intentional so reflections do not corrupt the sphere wake.
- Core solver implementation: `les_sphere_flow.py`.